# Feature Extraction

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import spacy
import spacy
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import MultiLabelBinarizer
df = pd.read_csv("../task2/transcribed_data_whisper.csv")
df.head()

,Sentence
0,Avem triburi noi!
1,"Am fost bine, nu m-a plouat deloc puțin pe pic..."
2,Pe colegii mei din baracă am înțeles că a plou...
3,"Suntem aici una cu natura, adică a ieșit cumva..."
4,"Mie îmi place foarte tare natura, îmi place fo..."


#### Part-of-Speech (POS) Tagging

In [5]:
language_model = "ro_core_news_lg"  
nlp = spacy.load(language_model)

def extract_pos_tags(sentence):
    doc = nlp(sentence)
    # You can return a list or join them into a string
    return [f"{token.text}_{token.pos_}" for token in doc]

df["POS_Tags"] = df["Sentence"].apply(extract_pos_tags)

In [6]:
df

,Sentence,POS_Tags
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]"
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,..."
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac..."
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu..."
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta..."
...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu..."
618,În deficit de număr și luptând împotriva numer...,"[În_ADP, deficit_NOUN, de_ADP, număr_NOUN, și_..."
619,"Bravo zeițe, sunteți mai puțini.","[Bravo_ADV, zeițe_NOUN, ,_PUNCT, sunteți_AUX, ..."
620,Ați luptat și a contat această energie masculi...,"[Ați_AUX, luptat_VERB, și_CCONJ, a_AUX, contat..."


#### Term Frequency-Inverse Document Frequency (TF-IDF)

In [7]:
pd.set_option('display.max_columns', None)

In [8]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["Sentence"])

# If you want to store as dense vectors:
df["TF_IDF"] = list(tfidf_matrix)

In [9]:
df

,Sentence,POS_Tags,TF_IDF
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0..."
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0...."
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0..."
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0..."
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0..."
...,...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu...","(0, 603)\t0.29484199221854757\n (0, 693)\t0..."
618,În deficit de număr și luptând împotriva numer...,"[În_ADP, deficit_NOUN, de_ADP, număr_NOUN, și_...","(0, 43)\t0.30372740179979335\n (0, 1251)\t0..."
619,"Bravo zeițe, sunteți mai puțini.","[Bravo_ADV, zeițe_NOUN, ,_PUNCT, sunteți_AUX, ...","(0, 913)\t0.4897353792264053\n (0, 1053)\t0..."
620,Ați luptat și a contat această energie masculi...,"[Ați_AUX, luptat_VERB, și_CCONJ, a_AUX, contat...","(0, 478)\t0.3132699017641063\n (0, 273)\t0...."


#### Pretrained Word Embeddings

In [10]:
language_model = "ro_core_news_lg"  
def get_pretrained_embedding(sentence):
    doc = nlp(sentence)
    # Filter out tokens without a vector (or zero vectors)
    vectors = [token.vector for token in doc if token.has_vector]
    if vectors:
        return sum(vectors) / len(vectors)
    else:
        # Return a zero vector of the same dimension if no vectors are available
        return [0] * nlp.vocab.vectors_length

df["Pretrained_Embeddings"] = df["Sentence"].apply(get_pretrained_embedding)

In [11]:
df

,Sentence,POS_Tags,TF_IDF,Pretrained_Embeddings
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0...","[1.377695, 1.0777285, -0.65787995, 1.95885, -1..."
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0....","[0.72625834, -1.9864999, 0.36522153, -0.839616..."
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0...","[0.62630403, -0.18003584, 0.83672255, -0.61534..."
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0...","[0.65965164, 0.4346206, 0.53087664, -0.2594880..."
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0...","[0.30756238, -0.9422255, 1.2564049, -0.2166265..."
...,...,...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu...","(0, 603)\t0.29484199221854757\n (0, 693)\t0...","[0.6019194, 1.6374106, 0.8823618, -0.42062333,..."
618,În deficit de număr și luptând împotriva numer...,"[În_ADP, deficit_NOUN, de_ADP, număr_NOUN, și_...","(0, 43)\t0.30372740179979335\n (0, 1251)\t0...","[-0.60048056, 1.1661624, 0.30852485, -1.431608..."
619,"Bravo zeițe, sunteți mai puțini.","[Bravo_ADV, zeițe_NOUN, ,_PUNCT, sunteți_AUX, ...","(0, 913)\t0.4897353792264053\n (0, 1053)\t0...","[-1.1221815, 0.24961714, 0.8885966, -0.6662972..."
620,Ați luptat și a contat această energie masculi...,"[Ați_AUX, luptat_VERB, și_CCONJ, a_AUX, contat...","(0, 478)\t0.3132699017641063\n (0, 273)\t0....","[-0.60728705, 1.6062807, 0.2803106, -1.9332379..."


### Sentiment Analysis

In [12]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialize VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Function to get sentiment score
def get_vader_sentiment(sentence):
    return analyzer.polarity_scores(str(sentence))["compound"]  # Convert to string in case of NaN

# Apply the function to each sentence
df["Sentiment_Score"] = df["Sentence"].apply(get_vader_sentiment)

# Show updated dataset
df.head()



,Sentence,POS_Tags,TF_IDF,Pretrained_Embeddings,Sentiment_Score
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0...","[1.377695, 1.0777285, -0.65787995, 1.95885, -1...",0.0000
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0....","[0.72625834, -1.9864999, 0.36522153, -0.839616...",0.0000
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0...","[0.62630403, -0.18003584, 0.83672255, -0.61534...",0.0000
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0...","[0.65965164, 0.4346206, 0.53087664, -0.2594880...",0.0000
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0...","[0.30756238, -0.9422255, 1.2564049, -0.2166265...",0.5719


#### Pretrained Word Embeddings

In [13]:
import gensim.downloader as api

# Load Google's pretrained Word2Vec model
word2vec_model = api.load("word2vec-google-news-300")  


In [14]:
import numpy as np

def sentence_to_avg_vector(sentence, model):
    words = str(sentence).split()  # Tokenize sentence
    word_vectors = [model[word] for word in words if word in model]  # Get vectors if word exists in model

    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)  # Return a zero vector if no words are found

    return np.mean(word_vectors, axis=0)  # Compute average vector


In [15]:
df["Custom_Embeddings"] = df["Sentence"].apply(lambda x: sentence_to_avg_vector(x, word2vec_model))


In [16]:
df

,Sentence,POS_Tags,TF_IDF,Pretrained_Embeddings,Sentiment_Score,Custom_Embeddings
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0...","[1.377695, 1.0777285, -0.65787995, 1.95885, -1...",0.0000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0....","[0.72625834, -1.9864999, 0.36522153, -0.839616...",0.0000,"[0.06039238, 0.07092285, 0.064834595, 0.183654..."
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0...","[0.62630403, -0.18003584, 0.83672255, -0.61534...",0.0000,"[-0.04273071, 0.058099367, 0.097631834, 0.0255..."
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0...","[0.65965164, 0.4346206, 0.53087664, -0.2594880...",0.0000,"[-0.03435262, 0.16389973, 0.0764974, 0.0268554..."
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0...","[0.30756238, -0.9422255, 1.2564049, -0.2166265...",0.5719,"[0.004638672, 0.12923177, 0.11063639, -0.00956..."
...,...,...,...,...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu...","(0, 603)\t0.29484199221854757\n (0, 693)\t0...","[0.6019194, 1.6374106, 0.8823618, -0.42062333,...",0.0000,"[-0.017285157, 0.120837405, 0.15463868, 0.0120..."
618,În deficit de număr și luptând împotriva numer...,"[În_ADP, deficit_NOUN, de_ADP, număr_NOUN, și_...","(0, 43)\t0.30372740179979335\n (0, 1251)\t0...","[-0.60048056, 1.1661624, 0.30852485, -1.431608...",-0.4019,"[0.025256347, 0.053344727, 0.1427246, 0.031860..."
619,"Bravo zeițe, sunteți mai puțini.","[Bravo_ADV, zeițe_NOUN, ,_PUNCT, sunteți_AUX, ...","(0, 913)\t0.4897353792264053\n (0, 1053)\t0...","[-1.1221815, 0.24961714, 0.8885966, -0.6662972...",0.0000,"[-0.21044922, 0.17895508, 0.2944336, -0.062484..."
620,Ați luptat și a contat această energie masculi...,"[Ați_AUX, luptat_VERB, și_CCONJ, a_AUX, contat...","(0, 478)\t0.3132699017641063\n (0, 273)\t0....","[-0.60728705, 1.6062807, 0.2803106, -1.9332379...",0.0000,"[-0.12796211, 0.13665771, 0.24975586, 0.061828..."


#### Bigrams

In [17]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Initialize CountVectorizer for bigrams (or trigrams by setting ngram_range=(3,3))
vectorizer = CountVectorizer(ngram_range=(2,2), token_pattern=r'\b\w+\b')  # Extracts word pairs (bigrams)

# Fit and transform the 'Sentence' column
X = vectorizer.fit_transform(df["Sentence"])

# Convert sparse matrix to DataFrame
ngrams_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

# Add a new column with extracted n-grams (list of n-grams for each sentence)
df["Bi_Grams"] = df["Sentence"].apply(lambda x: ", ".join(vectorizer.build_analyzer()(x)))


In [18]:
df

,Sentence,POS_Tags,TF_IDF,Pretrained_Embeddings,Sentiment_Score,Custom_Embeddings,Bi_Grams
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0...","[1.377695, 1.0777285, -0.65787995, 1.95885, -1...",0.0000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","avem triburi, triburi noi"
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0....","[0.72625834, -1.9864999, 0.36522153, -0.839616...",0.0000,"[0.06039238, 0.07092285, 0.064834595, 0.183654...","am fost, fost bine, bine nu, nu m, m a, a plou..."
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0...","[0.62630403, -0.18003584, 0.83672255, -0.61534...",0.0000,"[-0.04273071, 0.058099367, 0.097631834, 0.0255...","pe colegii, colegii mei, mei din, din baracă, ..."
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0...","[0.65965164, 0.4346206, 0.53087664, -0.2594880...",0.0000,"[-0.03435262, 0.16389973, 0.0764974, 0.0268554...","suntem aici, aici una, una cu, cu natura, natu..."
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0...","[0.30756238, -0.9422255, 1.2564049, -0.2166265...",0.5719,"[0.004638672, 0.12923177, 0.11063639, -0.00956...","mie îmi, îmi place, place foarte, foarte tare,..."
...,...,...,...,...,...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu...","(0, 603)\t0.29484199221854757\n (0, 693)\t0...","[0.6019194, 1.6374106, 0.8823618, -0.42062333,...",0.0000,"[-0.017285157, 0.120837405, 0.15463868, 0.0120...","aici te, te lupți, lupți la, la propriu, propr..."
618,În deficit de număr și luptând împotriva numer...,"[În_ADP, deficit_NOUN, de_ADP, număr_NOUN, și_...","(0, 43)\t0.30372740179979335\n (0, 1251)\t0...","[-0.60048056, 1.1661624, 0.30852485, -1.431608...",-0.4019,"[0.025256347, 0.053344727, 0.1427246, 0.031860...","în deficit, deficit de, de număr, număr și, și..."
619,"Bravo zeițe, sunteți mai puțini.","[Bravo_ADV, zeițe_NOUN, ,_PUNCT, sunteți_AUX, ...","(0, 913)\t0.4897353792264053\n (0, 1053)\t0...","[-1.1221815, 0.24961714, 0.8885966, -0.6662972...",0.0000,"[-0.21044922, 0.17895508, 0.2944336, -0.062484...","bravo zeițe, zeițe sunteți, sunteți mai, mai p..."
620,Ați luptat și a contat această energie masculi...,"[Ați_AUX, luptat_VERB, și_CCONJ, a_AUX, contat...","(0, 478)\t0.3132699017641063\n (0, 273)\t0....","[-0.60728705, 1.6062807, 0.2803106, -1.9332379...",0.0000,"[-0.12796211, 0.13665771, 0.24975586, 0.061828...","ați luptat, luptat și, și a, a contat, contat ..."


#### Trigrams

In [19]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Initialize CountVectorizer for trigrams
vectorizer = CountVectorizer(ngram_range=(3,3), token_pattern=r'\b\w+\b')

# Fit and transform the 'Sentence' column
X = vectorizer.fit_transform(df["Sentence"])

# Convert sparse matrix to DataFrame
ngrams_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

# Add a new column with extracted trigrams (list of trigrams for each sentence)
df["Tri_Grams"] = df["Sentence"].apply(lambda x: ", ".join(vectorizer.build_analyzer()(x)))

# Save (optional)
df.to_csv("transcribed_data_with_trigrams.csv", index=False)


In [20]:
df

,Sentence,POS_Tags,TF_IDF,Pretrained_Embeddings,Sentiment_Score,Custom_Embeddings,Bi_Grams,Tri_Grams
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0...","[1.377695, 1.0777285, -0.65787995, 1.95885, -1...",0.0000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","avem triburi, triburi noi",avem triburi noi
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0....","[0.72625834, -1.9864999, 0.36522153, -0.839616...",0.0000,"[0.06039238, 0.07092285, 0.064834595, 0.183654...","am fost, fost bine, bine nu, nu m, m a, a plou...","am fost bine, fost bine nu, bine nu m, nu m a,..."
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0...","[0.62630403, -0.18003584, 0.83672255, -0.61534...",0.0000,"[-0.04273071, 0.058099367, 0.097631834, 0.0255...","pe colegii, colegii mei, mei din, din baracă, ...","pe colegii mei, colegii mei din, mei din barac..."
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0...","[0.65965164, 0.4346206, 0.53087664, -0.2594880...",0.0000,"[-0.03435262, 0.16389973, 0.0764974, 0.0268554...","suntem aici, aici una, una cu, cu natura, natu...","suntem aici una, aici una cu, una cu natura, c..."
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0...","[0.30756238, -0.9422255, 1.2564049, -0.2166265...",0.5719,"[0.004638672, 0.12923177, 0.11063639, -0.00956...","mie îmi, îmi place, place foarte, foarte tare,...","mie îmi place, îmi place foarte, place foarte ..."
...,...,...,...,...,...,...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu...","(0, 603)\t0.29484199221854757\n (0, 693)\t0...","[0.6019194, 1.6374106, 0.8823618, -0.42062333,...",0.0000,"[-0.017285157, 0.120837405, 0.15463868, 0.0120...","aici te, te lupți, lupți la, la propriu, propr...","aici te lupți, te lupți la, lupți la propriu, ..."
618,În deficit de număr și luptând împotriva numer...,"[În_ADP, deficit_NOUN, de_ADP, număr_NOUN, și_...","(0, 43)\t0.30372740179979335\n (0, 1251)\t0...","[-0.60048056, 1.1661624, 0.30852485, -1.431608...",-0.4019,"[0.025256347, 0.053344727, 0.1427246, 0.031860...","în deficit, deficit de, de număr, număr și, și...","în deficit de, deficit de număr, de număr și, ..."
619,"Bravo zeițe, sunteți mai puțini.","[Bravo_ADV, zeițe_NOUN, ,_PUNCT, sunteți_AUX, ...","(0, 913)\t0.4897353792264053\n (0, 1053)\t0...","[-1.1221815, 0.24961714, 0.8885966, -0.6662972...",0.0000,"[-0.21044922, 0.17895508, 0.2944336, -0.062484...","bravo zeițe, zeițe sunteți, sunteți mai, mai p...","bravo zeițe sunteți, zeițe sunteți mai, sunteț..."
620,Ați luptat și a contat această energie masculi...,"[Ați_AUX, luptat_VERB, și_CCONJ, a_AUX, contat...","(0, 478)\t0.3132699017641063\n (0, 273)\t0....","[-0.60728705, 1.6062807, 0.2803106, -1.9332379...",0.0000,"[-0.12796211, 0.13665771, 0.24975586, 0.061828...","ați luptat, luptat și, și a, a contat, contat ...","ați luptat și, luptat și a, și a contat, a con..."


#### Dependency Parsing

In [21]:
import spacy
import pandas as pd

# Load Romanian SpaCy model
language_model = "ro_core_news_lg"  
nlp = spacy.load(language_model)

# Define function to extract dependency relations
def extract_dependency_tags(sentence):
    doc = nlp(sentence)
    return [f"{token.text}({token.dep_}→{token.head.text})" for token in doc]

# Apply dependency parsing
df["Dependency_Tags"] = df["Sentence"].apply(extract_dependency_tags)

In [22]:
df

,Sentence,POS_Tags,TF_IDF,Pretrained_Embeddings,Sentiment_Score,Custom_Embeddings,Bi_Grams,Tri_Grams,Dependency_Tags
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0...","[1.377695, 1.0777285, -0.65787995, 1.95885, -1...",0.0000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","avem triburi, triburi noi",avem triburi noi,"[Avem(ROOT→Avem), triburi(obj→Avem), noi(amod→..."
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0....","[0.72625834, -1.9864999, 0.36522153, -0.839616...",0.0000,"[0.06039238, 0.07092285, 0.064834595, 0.183654...","am fost, fost bine, bine nu, nu m, m a, a plou...","am fost bine, fost bine nu, bine nu m, nu m a,...","[Am(aux→bine), fost(cop→bine), bine(ROOT→bine)..."
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0...","[0.62630403, -0.18003584, 0.83672255, -0.61534...",0.0000,"[-0.04273071, 0.058099367, 0.097631834, 0.0255...","pe colegii, colegii mei, mei din, din baracă, ...","pe colegii mei, colegii mei din, mei din barac...","[Pe(case→colegii), colegii(obj→înțeles), mei(d..."
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0...","[0.65965164, 0.4346206, 0.53087664, -0.2594880...",0.0000,"[-0.03435262, 0.16389973, 0.0764974, 0.0268554...","suntem aici, aici una, una cu, cu natura, natu...","suntem aici una, aici una cu, una cu natura, c...","[Suntem(ROOT→Suntem), aici(advmod→Suntem), una..."
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0...","[0.30756238, -0.9422255, 1.2564049, -0.2166265...",0.5719,"[0.004638672, 0.12923177, 0.11063639, -0.00956...","mie îmi, îmi place, place foarte, foarte tare,...","mie îmi place, îmi place foarte, place foarte ...","[Mie(iobj→place), îmi(iobj→place), place(ROOT→..."
...,...,...,...,...,...,...,...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu...","(0, 603)\t0.29484199221854757\n (0, 693)\t0...","[0.6019194, 1.6374106, 0.8823618, -0.42062333,...",0.0000,"[-0.017285157, 0.120837405, 0.15463868, 0.0120...","aici te, te lupți, lupți la, la propriu, propr...","aici te lupți, te lupți la, lupți la propriu, ...","[Aici(advmod→lupți), te(obj→lupți), lupți(ROOT..."
618,În deficit de număr și luptând împotriva numer...,"[În_ADP, deficit_NOUN, de_ADP, număr_NOUN, și_...","(0, 43)\t0.30372740179979335\n (0, 1251)\t0...","[-0.60048056, 1.1661624, 0.30852485, -1.431608...",-0.4019,"[0.025256347, 0.053344727, 0.1427246, 0.031860...","în deficit, deficit de, de număr, număr și, și...","în deficit de, deficit de număr, de număr și, ...","[În(case→deficit), deficit(obl→adjudică), de(c..."
619,"Bravo zeițe, sunteți mai puțini.","[Bravo_ADV, zeițe_NOUN, ,_PUNCT, sunteți_AUX, ...","(0, 913)\t0.4897353792264053\n (0, 1053)\t0...","[-1.1221815, 0.24961714, 0.8885966, -0.6662972...",0.0000,"[-0.21044922, 0.17895508, 0.2944336, -0.062484...","bravo zeițe, zeițe sunteți, sunteți mai, mai p...","bravo zeițe sunteți, zeițe sunteți mai, sunteț...","[Bravo(advcl→puțini), zeițe(flat→Bravo), ,(pun..."
620,Ați luptat și a contat această energie masculi...,"[Ați_AUX, luptat_VERB, și_CCONJ, a_AUX, contat...","(0, 478)\t0.3132699017641063\n (0, 273)\t0....","[-0.60728705, 1.6062807, 0.2803106, -1.9332379...",0.0000,"[-0.12796211, 0.13665771, 0.24975586, 0.061828...","ați luptat, luptat și, și a, a contat, contat ...","ați luptat și, luptat și a, și a contat, a con...","[Ați(aux→luptat), luptat(ROOT→luptat), și(cc→c..."


#### Subjectivity score

In [23]:
from textblob import TextBlob
import pandas as pd

# Function to calculate subjectivity score using TextBlob
def subjectivity_score(sentence):
    blob = TextBlob(sentence)
    return blob.sentiment.subjectivity  # Return the subjectivity score

# Apply the subjectivity score to each sentence
df["Subjectivity_Score"] = df["Sentence"].apply(subjectivity_score)

In [24]:
df

,Sentence,POS_Tags,TF_IDF,Pretrained_Embeddings,Sentiment_Score,Custom_Embeddings,Bi_Grams,Tri_Grams,Dependency_Tags,Subjectivity_Score
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0...","[1.377695, 1.0777285, -0.65787995, 1.95885, -1...",0.0000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","avem triburi, triburi noi",avem triburi noi,"[Avem(ROOT→Avem), triburi(obj→Avem), noi(amod→...",0.0
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0....","[0.72625834, -1.9864999, 0.36522153, -0.839616...",0.0000,"[0.06039238, 0.07092285, 0.064834595, 0.183654...","am fost, fost bine, bine nu, nu m, m a, a plou...","am fost bine, fost bine nu, bine nu m, nu m a,...","[Am(aux→bine), fost(cop→bine), bine(ROOT→bine)...",0.0
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0...","[0.62630403, -0.18003584, 0.83672255, -0.61534...",0.0000,"[-0.04273071, 0.058099367, 0.097631834, 0.0255...","pe colegii, colegii mei, mei din, din baracă, ...","pe colegii mei, colegii mei din, mei din barac...","[Pe(case→colegii), colegii(obj→înțeles), mei(d...",0.0
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0...","[0.65965164, 0.4346206, 0.53087664, -0.2594880...",0.0000,"[-0.03435262, 0.16389973, 0.0764974, 0.0268554...","suntem aici, aici una, una cu, cu natura, natu...","suntem aici una, aici una cu, una cu natura, c...","[Suntem(ROOT→Suntem), aici(advmod→Suntem), una...",0.0
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0...","[0.30756238, -0.9422255, 1.2564049, -0.2166265...",0.5719,"[0.004638672, 0.12923177, 0.11063639, -0.00956...","mie îmi, îmi place, place foarte, foarte tare,...","mie îmi place, îmi place foarte, place foarte ...","[Mie(iobj→place), îmi(iobj→place), place(ROOT→...",1.0
...,...,...,...,...,...,...,...,...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu...","(0, 603)\t0.29484199221854757\n (0, 693)\t0...","[0.6019194, 1.6374106, 0.8823618, -0.42062333,...",0.0000,"[-0.017285157, 0.120837405, 0.15463868, 0.0120...","aici te, te lupți, lupți la, la propriu, propr...","aici te lupți, te lupți la, lupți la propriu, ...","[Aici(advmod→lupți), te(obj→lupți), lupți(ROOT...",0.0
618,În deficit de număr și luptând împotriva numer...,"[În_ADP, deficit_NOUN, de_ADP, număr_NOUN, și_...","(0, 43)\t0.30372740179979335\n (0, 1251)\t0...","[-0.60048056, 1.1661624, 0.30852485, -1.431608...",-0.4019,"[0.025256347, 0.053344727, 0.1427246, 0.031860...","în deficit, deficit de, de număr, număr și, și...","în deficit de, deficit de număr, de număr și, ...","[În(case→deficit), deficit(obl→adjudică), de(c...",0.0
619,"Bravo zeițe, sunteți mai puțini.","[Bravo_ADV, zeițe_NOUN, ,_PUNCT, sunteți_AUX, ...","(0, 913)\t0.4897353792264053\n (0, 1053)\t0...","[-1.1221815, 0.24961714, 0.8885966, -0.6662972...",0.0000,"[-0.21044922, 0.17895508, 0.2944336, -0.062484...","bravo zeițe, zeițe sunteți, sunteți mai, mai p...","bravo zeițe sunteți, zeițe sunteți mai, sunteț...","[Bravo(advcl→puțini), zeițe(flat→Bravo), ,(pun...",0.0
620,Ați luptat și a contat această energie masculi...,"[Ați_AUX, luptat_VERB, și_CCONJ, a_AUX, contat...","(0, 478)\t0.3132699017641063\n (0, 273)\t0....","[-0.60728705, 1.6062807, 0.2803106, -1.9332379...",0.0000,"[-0.12796211, 0.13665771, 0.24975586, 0.061828...","ați luptat, luptat și, și a, a contat, contat ...","ați luptat și, luptat și a, și a contat, a con...","[Ați(aux→luptat), luptat(ROOT→luptat), și(cc→c...",0.0


In [25]:
# Extract POS tags
pos_tags = [[word.split('_')[-1] for word in sentence] for sentence in df["POS_Tags"]]

# Tokenize and encode
tokenizer = Tokenizer()
tokenizer.fit_on_texts(pos_tags)
encoded_sequences = tokenizer.texts_to_sequences(pos_tags)

# Pad sequences to ensure uniform length
padded_sequences = pad_sequences(encoded_sequences, padding='post')

# Convert to DataFrame
df["POS_Encoded"] = list(padded_sequences)
df["POS_Encoded"]

0      [7, 2, 12, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...
1      [3, 3, 6, 1, 8, 5, 3, 7, 6, 6, 4, 2, 1, 0, 0, ...
2      [4, 2, 9, 4, 2, 3, 7, 13, 3, 7, 11, 2, 3, 7, 2...
3      [7, 6, 5, 4, 2, 1, 6, 3, 7, 6, 2, 9, 12, 4, 5,...
4      [2, 5, 3, 6, 6, 2, 1, 5, 3, 6, 6, 12, 1, 3, 2,...
                             ...                        
617    [6, 5, 3, 4, 6, 4, 2, 2, 11, 6, 6, 4, 12, 2, 4...
618    [4, 2, 4, 2, 11, 3, 4, 2, 1, 2, 5, 7, 15, 2, 4...
619    [6, 2, 1, 3, 6, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...
620    [3, 7, 11, 3, 7, 9, 2, 12, 7, 4, 4, 2, 11, 4, ...
621    [5, 3, 7, 4, 2, 1, 7, 6, 4, 1, 0, 0, 0, 0, 0, ...
Name: POS_Encoded, Length: 622, dtype: object

In [26]:
df["POS_Encoded"][1]

array([3, 3, 6, 1, 8, 5, 3, 7, 6, 6, 4, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [27]:
from sklearn.preprocessing import MultiLabelBinarizer

# Extract POS tags from each entry
pos_tags = [ [word.split('_')[-1] for word in sentence] for sentence in df["POS_Tags"] ]

# One-hot encode
mlb = MultiLabelBinarizer()
encoded_pos = mlb.fit_transform(pos_tags)

# Convert to DataFrame
df_pos_encoded = pd.DataFrame(encoded_pos, columns=mlb.classes_)
df_pos_encoded

,,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,VERB,X
0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0
1,0,0,1,1,1,0,0,0,1,0,1,1,0,1,0,1,0
2,0,0,1,1,1,1,1,0,1,0,0,1,0,1,1,1,0
3,0,1,1,1,1,0,1,0,1,0,0,1,0,1,0,1,0
4,0,1,0,1,1,0,0,0,1,0,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617,0,1,1,1,1,1,0,0,1,0,0,1,0,1,0,0,0
618,0,0,1,0,1,1,1,0,1,1,0,1,0,1,0,1,0
619,0,0,0,1,1,0,0,0,1,0,0,1,0,1,0,0,0
620,0,1,1,0,1,1,1,0,1,0,0,0,0,1,0,1,0


In [28]:
df_pos_encoded[df_pos_encoded['']!=0] 

,,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,VERB,X
119,1,1,1,1,1,0,1,0,1,0,0,1,0,1,0,1,0
246,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
255,1,1,1,0,0,1,1,0,1,0,0,0,0,1,0,0,0
256,1,1,1,1,1,1,1,0,1,0,0,0,0,1,0,0,0
258,1,1,0,0,1,1,1,0,1,0,0,1,0,1,0,0,0
262,1,0,0,1,0,0,0,0,1,0,0,0,0,1,0,1,0
268,1,0,1,1,1,1,1,0,1,0,1,0,0,1,0,0,0
269,1,1,1,0,1,1,1,0,1,0,0,1,1,1,0,0,0
270,1,0,1,1,0,1,1,1,1,0,0,0,0,1,0,1,0
271,1,1,0,1,1,0,1,0,1,0,1,1,0,1,1,1,0


In [29]:
df.iloc[284]['POS_Tags']

['Eu_PRON',
 'de_ADP',
 'dimineață_NOUN',
 'nu_PART',
 'm-_PRON',
 'ați_AUX',
 'simțit_VERB',
 'foarte_ADV',
 'bine_ADV',
 ',_PUNCT',
 'dar_CCONJ',
 'ați_AUX',
 'văzut_VERB',
 'toți_PRON',
 ',_PUNCT',
 'de_ADP',
 'asta_PRON',
 'mă_PRON',
 'retrag_AUX',
 ',_PUNCT',
 'faceți_VERB',
 'voi_VERB',
 'primele_NUM',
 '3_',
 'echipe_NOUN',
 '._PUNCT']

In [30]:
df_pos_encoded[df_pos_encoded['X']!=0] 

,,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,VERB,X
97,0,1,1,0,1,0,0,0,1,1,0,0,0,1,0,0,1
568,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,1,1
608,0,0,1,0,1,1,0,0,1,0,0,1,0,1,0,1,1


In [31]:
df.iloc[608]['POS_Tags']

['Dar_CCONJ',
 'discuția_NOUN',
 'despre_ADP',
 'ce_PRON',
 'am_AUX',
 'greșit_VERB',
 'e_AUX',
 'în_ADP',
 'principiu_NOUN',
 'low-value_X',
 '._PUNCT']

In [32]:
df_pos_encoded.rename(columns={'': 'NUMBER'}, inplace=True)

In [33]:
df = pd.concat([df, df_pos_encoded], axis=1)

In [34]:
df["POS_Vector"] = df_pos_encoded.apply(lambda row: np.array(row.tolist()), axis=1)
df

,Sentence,POS_Tags,TF_IDF,Pretrained_Embeddings,Sentiment_Score,Custom_Embeddings,Bi_Grams,Tri_Grams,Dependency_Tags,Subjectivity_Score,POS_Encoded,NUMBER,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,VERB,X,POS_Vector
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0...","[1.377695, 1.0777285, -0.65787995, 1.95885, -1...",0.0000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","avem triburi, triburi noi",avem triburi noi,"[Avem(ROOT→Avem), triburi(obj→Avem), noi(amod→...",0.0,"[7, 2, 12, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,"[0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, ..."
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0....","[0.72625834, -1.9864999, 0.36522153, -0.839616...",0.0000,"[0.06039238, 0.07092285, 0.064834595, 0.183654...","am fost, fost bine, bine nu, nu m, m a, a plou...","am fost bine, fost bine nu, bine nu m, nu m a,...","[Am(aux→bine), fost(cop→bine), bine(ROOT→bine)...",0.0,"[3, 3, 6, 1, 8, 5, 3, 7, 6, 6, 4, 2, 1, 0, 0, ...",0,0,1,1,1,0,0,0,1,0,1,1,0,1,0,1,0,"[0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, ..."
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0...","[0.62630403, -0.18003584, 0.83672255, -0.61534...",0.0000,"[-0.04273071, 0.058099367, 0.097631834, 0.0255...","pe colegii, colegii mei, mei din, din baracă, ...","pe colegii mei, colegii mei din, mei din barac...","[Pe(case→colegii), colegii(obj→înțeles), mei(d...",0.0,"[4, 2, 9, 4, 2, 3, 7, 13, 3, 7, 11, 2, 3, 7, 2...",0,0,1,1,1,1,1,0,1,0,0,1,0,1,1,1,0,"[0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, ..."
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0...","[0.65965164, 0.4346206, 0.53087664, -0.2594880...",0.0000,"[-0.03435262, 0.16389973, 0.0764974, 0.0268554...","suntem aici, aici una, una cu, cu natura, natu...","suntem aici una, aici una cu, una cu natura, c...","[Suntem(ROOT→Suntem), aici(advmod→Suntem), una...",0.0,"[7, 6, 5, 4, 2, 1, 6, 3, 7, 6, 2, 9, 12, 4, 5,...",0,1,1,1,1,0,1,0,1,0,0,1,0,1,0,1,0,"[0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, ..."
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0...","[0.30756238, -0.9422255, 1.2564049, -0.2166265...",0.5719,"[0.004638672, 0.12923177, 0.11063639, -0.00956...","mie îmi, îmi place, place foarte, foarte tare,...","mie îmi place, îmi place foarte, place foarte ...","[Mie(iobj→place), îmi(iobj→place), place(ROOT→...",1.0,"[2, 5, 3, 6, 6, 2, 1, 5, 3, 6, 6, 12, 1, 3, 2,...",0,1,0,1,1,0,0,0,1,0,0,1,0,1,0,0,0,"[0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu...","(0, 603)\t0.29484199221854757\n (0, 693)\t0...","[0.6019194, 1.6374106, 0.8823618, -0.42062333,...",0.0000,"[-0.017285157, 0.120837405, 0.15463868, 0.0120...","aici te, te lupți, lupți la, la propriu, propr...","aici te lupți, te lupți la, lupți la propriu, ...","[Aici(advmod→lupți), te(obj→lupți), lupți(ROOT...",0.0,"[6, 5, 3, 4, 6, 4, 2, 2, 11, 6, 6, 4, 12, 2, 4...",0,1,1,1,1,1,0,0,1,0,0,1,0,1,0,0,0,"[0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, ..."
618,În deficit de număr și luptând împotriva numer...,"[În_ADP, deficit_NOUN, de_ADP, număr_NOUN, și_...","(0, 43)\t0.30372740179979335\n (0, 1251)\t0...","[-0.60048056, 1.1661624, 0.30852485, -1.431608...",-0.4019,"[0.025256347, 0.053344727, 0.1427246, 0.031860...","în deficit, deficit de, de număr, număr și, și...","în deficit de, deficit de număr, de 

In [ ]:
romanian_stopwords = ["a","abia","acea","aceasta","această","aceea","aceeasi","acei","aceia","acel","acela","acelasi","acele","acelea","acest","acesta","aceste","acestea","acestei","acestia","acestui","aceşti","aceştia","acolo","acord","acum","adica","ai","aia","aibă","aici","aiurea","al","ala","alaturi","ale","alea","alt","alta","altceva","altcineva","alte","altfel","alti","altii","altul","am","anume","apoi","ar","are","as","asa","asemenea","asta","astazi","astea","astfel","astăzi","asupra","atare","atat","atata","atatea","atatia","ati","atit","atita","atitea","atitia","atunci","au","avea","avem","aveţi","avut","azi","aş","aşadar","aţi","b","ba","bine","bucur","bună","c","ca","cam","cand","capat","care","careia","carora","caruia","cat","catre","caut","ce","cea","ceea","cei","ceilalti","cel","cele","celor","ceva","chiar","ci","cinci","cind","cine","cineva","cit","cita","cite","citeva","citi","citiva","conform","contra","cu","cui","cum","cumva","curând","curînd","când","cât","câte","câtva","câţi","cînd","cît","cîte","cîtva","cîţi","că","căci","cărei","căror","cărui","către","d","da","daca","dacă","dar","dat","datorită","dată","dau","de","deasupra","deci","decit","degraba","deja","deoarece","departe","desi","despre","deşi","din","dinaintea","dintr","dintr-","dintre","doar","doi","doilea","două","drept","dupa","după","dă","e","ea","ei","el","ele","era","eram","este","eu","exact","eşti","f","face","fara","fata","fel","fi","fie","fiecare","fii","fim","fiu","fiţi","foarte","fost","frumos","fără","g","geaba","graţie","h","halbă","i","ia","iar","ieri","ii","il","imi","in","inainte","inapoi","inca","incit","insa","intr","intre","isi","iti","j","k","l","la","le","li","lor","lui","lângă","lîngă","m","ma","mai","mare","mea","mei","mele","mereu","meu","mi","mie","mine","mod","mult","multa","multe","multi","multă","mulţi","mulţumesc","mâine","mîine","mă","n","ne","nevoie","ni","nici","niciodata","nicăieri","nimeni","nimeri","nimic","niste","nişte","noastre","noastră","noi","noroc","nostri","nostru","nou","noua","nouă","noştri","nu","numai","o","opt","or","ori","oricare","orice","oricine","oricum","oricând","oricât","oricînd","oricît","oriunde","p","pai","parca","patra","patru","patrulea","pe","pentru","peste","pic","pina","plus","poate","pot","prea","prima","primul","prin","printr-","putini","puţin","puţina","puţină","până","pînă","r","rog","s","sa","sa-mi","sa-ti","sai","sale","sau","se","si","sint","sintem","spate","spre","sub","sunt","suntem","sunteţi","sus","sută","sînt","sîntem","sînteţi","să","săi","său","t","ta","tale","te","ti","timp","tine","toata","toate","toată","tocmai","tot","toti","totul","totusi","totuşi","toţi","trei","treia","treilea","tu","tuturor","tăi","tău","u","ul","ului","un","una","unde","undeva","unei","uneia","unele","uneori","unii","unor","unora","unu","unui","unuia","unul","v","va","vi","voastre","voastră","voi","vom","vor","vostru","vouă","voştri","vreme","vreo","vreun","vă","x","z","zece","zero","zi","zice","îi","îl","îmi","împotriva","în","înainte","înaintea","încotro","încât","încît","între","întrucât","întrucît","îţi","ăla","ălea","ăsta","ăstea","ăştia","şapte","şase","şi","ştiu","ţi","ţie"]
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
vectorizer = TfidfVectorizer(stop_words=romanian_stopwords)
tfidf_matrix = vectorizer.fit_transform(df["Sentence"])
tfidf_matrix

c:\Users\Kira\anaconda3\envs\block_c\lib\site-packages\sklearn\feature_extraction\text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['printr'] not in stop_words.
  warnings.warn(


<622x1093 sparse matrix of type '<class 'numpy.float64'>'
	with 2230 stored elements in Compressed Sparse Row format>

In [36]:
print(tfidf_matrix)

  (0, 939)	1.0
  (1, 661)	0.5369981081583747
  (1, 748)	0.43720705435770174
  (1, 257)	0.5369981081583747
  (1, 676)	0.4817842414281673
  (2, 517)	0.3819732613892467
  (2, 386)	0.3819732613892467
  (2, 17)	0.3819732613892467
  (2, 1079)	0.15543190327013237
  (2, 1072)	0.358999273059349
  (2, 98)	0.3819732613892467
  (2, 186)	0.3819732613892467
  (2, 676)	0.34269896893191
  (3, 552)	0.4618428118574809
  (3, 851)	0.4618428118574809
  (3, 445)	0.4618428118574809
  (3, 25)	0.4143563736805282
  (3, 596)	0.43406502623115006
  (4, 654)	0.2678262386036487
  (4, 511)	0.2985198997356087
  (4, 548)	0.2985198997356087
  (4, 898)	0.5611305179271392
  (4, 669)	0.5970397994712174
  (4, 596)	0.2805652589635696
  (5, 590)	0.5304111124897462
  :	:
  (618, 527)	0.351618262931242
  (618, 255)	0.351618262931242
  (618, 1073)	0.3304699923958095
  (618, 819)	0.31546505565836486
  (618, 616)	0.31546505565836486
  (618, 1030)	0.28627658581843873
  (618, 986)	0.3304699923958095
  (618, 617)	0.3304699923958095
 

In [38]:
feature_names = vectorizer.get_feature_names_out()

# Convert sparse matrix to a list of dictionaries
tfidf_dicts = []
for row in tfidf_matrix:
    row_dict = {feature_names[i]: row[0, i] for i in row.nonzero()[1]}
    tfidf_dicts.append(row_dict)

# Store in the DataFrame
df["TF_IDF_V2"] = tfidf_dicts

In [39]:
df

,Sentence,POS_Tags,TF_IDF,Pretrained_Embeddings,Sentiment_Score,Custom_Embeddings,Bi_Grams,Tri_Grams,Dependency_Tags,Subjectivity_Score,POS_Encoded,NUMBER,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,VERB,X,POS_Vector,TF_IDF_V2
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0...","[1.377695, 1.0777285, -0.65787995, 1.95885, -1...",0.0000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","avem triburi, triburi noi",avem triburi noi,"[Avem(ROOT→Avem), triburi(obj→Avem), noi(amod→...",0.0,"[7, 2, 12, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,"[0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, ...",{'triburi': 1.0}
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0....","[0.72625834, -1.9864999, 0.36522153, -0.839616...",0.0000,"[0.06039238, 0.07092285, 0.064834595, 0.183654...","am fost, fost bine, bine nu, nu m, m a, a plou...","am fost bine, fost bine nu, bine nu m, nu m a,...","[Am(aux→bine), fost(cop→bine), bine(ROOT→bine)...",0.0,"[3, 3, 6, 1, 8, 5, 3, 7, 6, 6, 4, 2, 1, 0, 0, ...",0,0,1,1,1,0,0,0,1,0,1,1,0,1,0,1,0,"[0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, ...","{'picioare': 0.5369981081583747, 'puțin': 0.43..."
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0...","[0.62630403, -0.18003584, 0.83672255, -0.61534...",0.0000,"[-0.04273071, 0.058099367, 0.097631834, 0.0255...","pe colegii, colegii mei, mei din, din baracă, ...","pe colegii mei, colegii mei din, mei din barac...","[Pe(case→colegii), colegii(obj→înțeles), mei(d...",0.0,"[4, 2, 9, 4, 2, 3, 7, 13, 3, 7, 11, 2, 3, 7, 2...",0,0,1,1,1,1,1,0,1,0,0,1,0,1,1,1,0,"[0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, ...","{'lucrat': 0.3819732613892467, 'fisuri': 0.381..."
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0...","[0.65965164, 0.4346206, 0.53087664, -0.2594880...",0.0000,"[-0.03435262, 0.16389973, 0.0764974, 0.0268554...","suntem aici, aici una, una cu, cu natura, natu...","suntem aici una, aici una cu, una cu natura, c...","[Suntem(ROOT→Suntem), aici(advmod→Suntem), una...",0.0,"[7, 6, 5, 4, 2, 1, 6, 3, 7, 6, 2, 9, 12, 4, 5,...",0,1,1,1,1,0,1,0,1,0,0,1,0,1,0,1,0,"[0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, ...","{'masculin': 0.4618428118574809, 'spiritul': 0..."
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0...","[0.30756238, -0.9422255, 1.2564049, -0.2166265...",0.5719,"[0.004638672, 0.12923177, 0.11063639, -0.00956...","mie îmi, îmi place, place foarte, foarte tare,...","mie îmi place, îmi place foarte, place foarte ...","[Mie(iobj→place), îmi(iobj→place), place(ROOT→...",1.0,"[2, 5, 3, 6, 6, 2, 1, 5, 3, 6, 6, 12, 1, 3, 2,...",0,1,0,1,1,0,0,0,1,0,0,1,0,1,0,0,0,"[0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, ...","{'perfect': 0.2678262386036487, 'locul': 0.298..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu...","(0, 603)\t0.29484199221854757\n (0, 693)\t0...","[0.6019194, 1.6374106, 0.8823618, -0.42062333,...",0.0000,"[-0.017285157, 0.120837405, 0.15463868, 0.0120...","aici te, te lupți, lupți la, la propriu, propr...","aici te lupți, te lupți la, lupți la propriu, ...","[Aici(advmod→lupți), te(obj→lupți), lupți(ROOT...",0.0,"[6, 5, 3, 4, 6, 4, 2, 2, 11, 6, 6, 4, 12, 2, 4...",0,1,1,1,1,1,0,0,1,0,0,1,0,1,0,0,0,"[0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, ...","{'julituri': 0.3555092544541718, 'mici': 0.355..."
618,În deficit de număr și luptând împotriva numer...,"[În_ADP, d

In [40]:
df["TF_IDF_V3"] = df["TF_IDF_V2"].apply(lambda x: str(x))

In [41]:
df

,Sentence,POS_Tags,TF_IDF,Pretrained_Embeddings,Sentiment_Score,Custom_Embeddings,Bi_Grams,Tri_Grams,Dependency_Tags,Subjectivity_Score,POS_Encoded,NUMBER,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,VERB,X,POS_Vector,TF_IDF_V2,TF_IDF_V3
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0...","[1.377695, 1.0777285, -0.65787995, 1.95885, -1...",0.0000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","avem triburi, triburi noi",avem triburi noi,"[Avem(ROOT→Avem), triburi(obj→Avem), noi(amod→...",0.0,"[7, 2, 12, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,"[0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, ...",{'triburi': 1.0},{'triburi': 1.0}
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0....","[0.72625834, -1.9864999, 0.36522153, -0.839616...",0.0000,"[0.06039238, 0.07092285, 0.064834595, 0.183654...","am fost, fost bine, bine nu, nu m, m a, a plou...","am fost bine, fost bine nu, bine nu m, nu m a,...","[Am(aux→bine), fost(cop→bine), bine(ROOT→bine)...",0.0,"[3, 3, 6, 1, 8, 5, 3, 7, 6, 6, 4, 2, 1, 0, 0, ...",0,0,1,1,1,0,0,0,1,0,1,1,0,1,0,1,0,"[0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, ...","{'picioare': 0.5369981081583747, 'puțin': 0.43...","{'picioare': 0.5369981081583747, 'puțin': 0.43..."
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0...","[0.62630403, -0.18003584, 0.83672255, -0.61534...",0.0000,"[-0.04273071, 0.058099367, 0.097631834, 0.0255...","pe colegii, colegii mei, mei din, din baracă, ...","pe colegii mei, colegii mei din, mei din barac...","[Pe(case→colegii), colegii(obj→înțeles), mei(d...",0.0,"[4, 2, 9, 4, 2, 3, 7, 13, 3, 7, 11, 2, 3, 7, 2...",0,0,1,1,1,1,1,0,1,0,0,1,0,1,1,1,0,"[0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, ...","{'lucrat': 0.3819732613892467, 'fisuri': 0.381...","{'lucrat': 0.3819732613892467, 'fisuri': 0.381..."
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0...","[0.65965164, 0.4346206, 0.53087664, -0.2594880...",0.0000,"[-0.03435262, 0.16389973, 0.0764974, 0.0268554...","suntem aici, aici una, una cu, cu natura, natu...","suntem aici una, aici una cu, una cu natura, c...","[Suntem(ROOT→Suntem), aici(advmod→Suntem), una...",0.0,"[7, 6, 5, 4, 2, 1, 6, 3, 7, 6, 2, 9, 12, 4, 5,...",0,1,1,1,1,0,1,0,1,0,0,1,0,1,0,1,0,"[0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, ...","{'masculin': 0.4618428118574809, 'spiritul': 0...","{'masculin': 0.4618428118574809, 'spiritul': 0..."
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0...","[0.30756238, -0.9422255, 1.2564049, -0.2166265...",0.5719,"[0.004638672, 0.12923177, 0.11063639, -0.00956...","mie îmi, îmi place, place foarte, foarte tare,...","mie îmi place, îmi place foarte, place foarte ...","[Mie(iobj→place), îmi(iobj→place), place(ROOT→...",1.0,"[2, 5, 3, 6, 6, 2, 1, 5, 3, 6, 6, 12, 1, 3, 2,...",0,1,0,1,1,0,0,0,1,0,0,1,0,1,0,0,0,"[0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, ...","{'perfect': 0.2678262386036487, 'locul': 0.298...","{'perfect': 0.2678262386036487, 'locul': 0.298..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu...","(0, 603)\t0.29484199221854757\n (0, 693)\t0...","[0.6019194, 1.6374106, 0.8823618, -0.42062333,...",0.0000,"[-0.017285157, 0.120837405, 0.15463868, 0.0120...","aici te, te lupți, lupți la, la propriu, propr...","aici te lupți, te lupți la, lupți la propriu, ...","[Aici(advmod→lupți), te(obj→lupți), lupți(ROOT...",0.0,"[6, 5, 3, 4, 6,

In [43]:
print(df.loc[0, "TF_IDF_V3"])  # Print TF-IDF for the first sentence

{'triburi': 1.0}


In [44]:
ro_stop = []
vectorizer = TfidfVectorizer(stop_words=ro_stop)
tfidf_matrix_ro = vectorizer.fit_transform(df["Sentence"])
print(tfidf_matrix_ro)

  (0, 755)	0.5667207640158156
  (0, 1125)	0.6432647726609333
  (0, 125)	0.5148184222488625
  (1, 819)	0.4450755737236425
  (1, 806)	0.21034707020931198
  (1, 912)	0.36236660352793676
  (1, 336)	0.4450755737236425
  (1, 834)	0.39931313426789644
  (1, 761)	0.16947733129969406
  (1, 145)	0.3077882553564043
  (1, 500)	0.30001207276588293
  (1, 88)	0.23525650487752456
  (2, 425)	0.22984444415429778
  (2, 606)	0.12336417192293746
  (2, 636)	0.28230567291027714
  (2, 326)	0.10717638291730937
  (2, 715)	0.21091692508860962
  (2, 657)	0.15100239889789763
  (2, 330)	0.21091692508860962
  (2, 487)	0.28230567291027714
  (2, 127)	0.23629972283085407
  (2, 30)	0.28230567291027714
  (2, 1293)	0.11487533940153202
  (2, 309)	0.16011987904280722
  (2, 1285)	0.26532624557728524
  :	:
  (619, 913)	0.4897353792264053
  (619, 1053)	0.4897353792264053
  (619, 154)	0.4897353792264053
  (619, 1232)	0.46027998005483856
  (619, 657)	0.2619544139017784
  (620, 478)	0.3132699017641063
  (620, 273)	0.33331746065163

In [ ]:
feature_names = vectorizer.get_feature_names_out()

# Convert sparse matrix to a list of dictionaries
tfidf_dicts = []
for row in tfidf_matrix_ro:
    row_dict = {feature_names[i]: row[0, i] for i in row.nonzero()[1]}
    tfidf_dicts.append(row_dict)

# Store in the DataFrame
df["TF_IDF_V4"] = tfidf_dicts

In [46]:
df

,Sentence,POS_Tags,TF_IDF,Pretrained_Embeddings,Sentiment_Score,Custom_Embeddings,Bi_Grams,Tri_Grams,Dependency_Tags,Subjectivity_Score,POS_Encoded,NUMBER,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,VERB,X,POS_Vector,TF_IDF_V2,TF_IDF_V3,TF_IDF_V4
0,Avem triburi noi!,"[Avem_VERB, triburi_NOUN, noi_ADJ, !_PUNCT]","(0, 755)\t0.5667207640158156\n (0, 1125)\t0...","[1.377695, 1.0777285, -0.65787995, 1.95885, -1...",0.0000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","avem triburi, triburi noi",avem triburi noi,"[Avem(ROOT→Avem), triburi(obj→Avem), noi(amod→...",0.0,"[7, 2, 12, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,"[0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, ...",{'triburi': 1.0},{'triburi': 1.0},"{'noi': 0.5667207640158156, 'triburi': 0.64326..."
1,"Am fost bine, nu m-a plouat deloc puțin pe pic...","[Am_AUX, fost_AUX, bine_ADV, ,_PUNCT, nu_PART,...","(0, 819)\t0.4450755737236425\n (0, 806)\t0....","[0.72625834, -1.9864999, 0.36522153, -0.839616...",0.0000,"[0.06039238, 0.07092285, 0.064834595, 0.183654...","am fost, fost bine, bine nu, nu m, m a, a plou...","am fost bine, fost bine nu, bine nu m, nu m a,...","[Am(aux→bine), fost(cop→bine), bine(ROOT→bine)...",0.0,"[3, 3, 6, 1, 8, 5, 3, 7, 6, 6, 4, 2, 1, 0, 0, ...",0,0,1,1,1,0,0,0,1,0,1,1,0,1,0,1,0,"[0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, ...","{'picioare': 0.5369981081583747, 'puțin': 0.43...","{'picioare': 0.5369981081583747, 'puțin': 0.43...","{'picioare': 0.4450755737236425, 'pe': 0.21034..."
2,Pe colegii mei din baracă am înțeles că a plou...,"[Pe_ADP, colegii_NOUN, mei_DET, din_ADP, barac...","(0, 425)\t0.22984444415429778\n (0, 606)\t0...","[0.62630403, -0.18003584, 0.83672255, -0.61534...",0.0000,"[-0.04273071, 0.058099367, 0.097631834, 0.0255...","pe colegii, colegii mei, mei din, din baracă, ...","pe colegii mei, colegii mei din, mei din barac...","[Pe(case→colegii), colegii(obj→înțeles), mei(d...",0.0,"[4, 2, 9, 4, 2, 3, 7, 13, 3, 7, 11, 2, 3, 7, 2...",0,0,1,1,1,1,1,0,1,0,0,1,0,1,1,1,0,"[0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, ...","{'lucrat': 0.3819732613892467, 'fisuri': 0.381...","{'lucrat': 0.3819732613892467, 'fisuri': 0.381...","{'el': 0.22984444415429778, 'la': 0.1233641719..."
3,"Suntem aici una cu natura, adică a ieșit cumva...","[Suntem_VERB, aici_ADV, una_PRON, cu_ADP, natu...","(0, 676)\t0.3201091277723238\n (0, 1288)\t0...","[0.65965164, 0.4346206, 0.53087664, -0.2594880...",0.0000,"[-0.03435262, 0.16389973, 0.0764974, 0.0268554...","suntem aici, aici una, una cu, cu natura, natu...","suntem aici una, aici una cu, una cu natura, c...","[Suntem(ROOT→Suntem), aici(advmod→Suntem), una...",0.0,"[7, 6, 5, 4, 2, 1, 6, 3, 7, 6, 2, 9, 12, 4, 5,...",0,1,1,1,1,0,1,0,1,0,0,1,0,1,0,1,0,"[0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, ...","{'masculin': 0.4618428118574809, 'spiritul': 0...","{'masculin': 0.4618428118574809, 'spiritul': 0...","{'masculin': 0.3201091277723238, 'ăsta': 0.239..."
4,"Mie îmi place foarte tare natura, îmi place fo...","[Mie_NOUN, îmi_PRON, place_AUX, foarte_ADV, ta...","(0, 810)\t0.21875167198502743\n (0, 629)\t0...","[0.30756238, -0.9422255, 1.2564049, -0.2166265...",0.5719,"[0.004638672, 0.12923177, 0.11063639, -0.00956...","mie îmi, îmi place, place foarte, foarte tare,...","mie îmi place, îmi place foarte, place foarte ...","[Mie(iobj→place), îmi(iobj→place), place(ROOT→...",1.0,"[2, 5, 3, 6, 6, 2, 1, 5, 3, 6, 6, 12, 1, 3, 2,...",0,1,0,1,1,0,0,0,1,0,0,1,0,1,0,0,0,"[0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, ...","{'perfect': 0.2678262386036487, 'locul': 0.298...","{'perfect': 0.2678262386036487, 'locul': 0.298...","{'perfect': 0.21875167198502743, 'locul': 0.24..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617,Aici te lupți la propriu cu sudoarea frunții ș...,"[Aici_ADV, te_PRON, lupți_AUX, la_ADP, propriu...","(0, 603)\t0.29484199221854757\n (0, 693)\t0...","[0.6019194, 